# spi_nb_gold_refresh_sql_endpoint

**Purpose.** Refreshes the SQL analytics endpoint metadata of `spi_lakehouse`
so that `dbo.spi_sp_gold_load` reads the latest Silver commits. Closes
ADR-014, point (b): the endpoint may lag behind Delta writes.

**Inputs.** None. The workspace ID and the SQL analytics endpoint ID of
`spi_lakehouse` are resolved at runtime.

**Outputs.** None written. Calls the Fabric REST API
`POST /v1/workspaces/{workspaceId}/sqlEndpoints/{sqlEndpointId}/refreshMetadata`
for the three Silver tables that `dbo.spi_sp_gold_load` reads, waits for the
long-running operation, and checks the per-table sync status.

**Dependencies.** `sempy` (semantic link, preinstalled). Runs on the Fabric
**Python** runtime (no Spark session), per ADR-012.

**Exit contract.** On success, calls `notebookutils.notebook.exit()` with
`{"tables_synced": <int>, "tables_not_run": <int>, "duration_seconds": <float>}`.
`NotRun` is not a failure. Raises, uncaught, if any table reports `Failure`,
a Silver table is missing from the result, the operation fails, or polling
times out (ADR-009, 2026-09-30 addendum).

**Last updated.** 2026-10-15.


In [ ]:
import json
import logging
import time

import sempy.fabric as fabric
from sempy.fabric.exceptions import FabricHTTPException

# ────────────────────────────────────────────────────────────────────────────
# Constants
# ────────────────────────────────────────────────────────────────────────────

LAKEHOUSE_NAME = "spi_lakehouse"

# The Silver tables dbo.spi_sp_gold_load reads (03-src/warehouse/spi_sp_gold_load.sql)
SILVER_SCHEMA = "dbo"
SILVER_TABLES = [
    "spi_silver_energy_cleaned",
    "spi_silver_construction_cleaned",
    "spi_silver_tax_cleaned",
]

# The service times the refresh out after 15 minutes by default (request body
# `timeout`, not sent). Polling waits longer, so a server-side timeout is
# reported by the service rather than by this loop.
POLL_TIMEOUT_SECONDS = 20 * 60

# Used when a response carries no Retry-After header
DEFAULT_RETRY_AFTER_SECONDS = 5

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Configure Logging
# ────────────────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
log = logging.getLogger("gold_refresh_sql_endpoint")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create function resolve_sql_endpoint_id
# ────────────────────────────────────────────────────────────────────────────

def resolve_sql_endpoint_id(client: fabric.FabricRestClient, workspace_id: str) -> str:
    """
    Resolve the SQL analytics endpoint ID of spi_lakehouse.
    Args:
        client (FabricRestClient): The Fabric REST client.
        workspace_id (str): The workspace ID.
    Returns:
        str: The SQL analytics endpoint ID.
    Raises:
        FabricHTTPException: If the Get Lakehouse call does not return 200.
        RuntimeError: If the endpoint is not provisioned.
    """
    lakehouse_id = fabric.resolve_item_id(
        LAKEHOUSE_NAME, item_type="Lakehouse", workspace=workspace_id
    )
    response = client.get(f"v1/workspaces/{workspace_id}/lakehouses/{lakehouse_id}")
    if response.status_code != 200:
        raise FabricHTTPException(response)

    sql_endpoint = response.json()["properties"]["sqlEndpointProperties"]
    if sql_endpoint["provisioningStatus"] != "Success":
        raise RuntimeError(
            f"SQL analytics endpoint of {LAKEHOUSE_NAME} is not provisioned: "
            f"{sql_endpoint['provisioningStatus']}"
        )
    log.info(f"{LAKEHOUSE_NAME}: lakehouse {lakehouse_id}, SQL endpoint {sql_endpoint['id']}")
    return sql_endpoint["id"]

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create function refresh_metadata
# ────────────────────────────────────────────────────────────────────────────

def refresh_metadata(
    client: fabric.FabricRestClient, workspace_id: str, sql_endpoint_id: str
) -> list[dict]:
    """
    Start the metadata refresh for the Silver tables and wait for it to finish.
    The API is a long-running operation: 200 returns the result at once,
    202 returns an operation ID to poll.
    Args:
        client (FabricRestClient): The Fabric REST client.
        workspace_id (str): The workspace ID.
        sql_endpoint_id (str): The SQL analytics endpoint ID.
    Returns:
        list[dict]: One sync status per table (tableName, status, error, ...).
    Raises:
        FabricHTTPException: On an unexpected HTTP status.
        RuntimeError: If the operation reports Failed.
        TimeoutError: If the operation is still running after POLL_TIMEOUT_SECONDS.
    """
    response = client.post(
        f"v1/workspaces/{workspace_id}/sqlEndpoints/{sql_endpoint_id}/refreshMetadata",
        json={"tables": [{"schema": SILVER_SCHEMA, "tableNames": SILVER_TABLES}]},
    )
    if response.status_code == 200:
        return response.json()["value"]
    if response.status_code != 202:
        raise FabricHTTPException(response)

    # Poll by operation ID (Get Operation State), then read Get Operation Result
    operation_id = response.headers["x-ms-operation-id"]
    log.info(f"Refresh accepted, operation {operation_id}")
    deadline = time.monotonic() + POLL_TIMEOUT_SECONDS

    while True:
        time.sleep(int(response.headers.get("Retry-After", DEFAULT_RETRY_AFTER_SECONDS)))
        response = client.get(f"v1/operations/{operation_id}")
        if response.status_code != 200:
            raise FabricHTTPException(response)

        state = response.json()
        status = state["status"]
        log.info(f"Operation {operation_id}: {status} ({state.get('percentComplete')}%)")
        if status == "Succeeded":
            break
        if status == "Failed":
            raise RuntimeError(f"Refresh operation {operation_id} failed: {state.get('error')}")
        if time.monotonic() > deadline:
            raise TimeoutError(
                f"Refresh operation {operation_id} still {status} "
                f"after {POLL_TIMEOUT_SECONDS} s"
            )

    response = client.get(f"v1/operations/{operation_id}/result")
    if response.status_code != 200:
        raise FabricHTTPException(response)
    return response.json()["value"]

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create function check_tables
# ────────────────────────────────────────────────────────────────────────────

def check_tables(tables: list[dict]) -> None:
    """
    Raise if a Silver table is missing from the result or failed to sync.
    Args:
        tables (list[dict]): The per-table sync statuses.
    Raises:
        RuntimeError: If a Silver table is missing or has status Failure.
    """
    # A schema-enabled lakehouse prefixes tableName with the schema ("dbo.<table>")
    returned = {t["tableName"].removeprefix(f"{SILVER_SCHEMA}.") for t in tables}
    missing = sorted(set(SILVER_TABLES) - returned)
    if missing:
        raise RuntimeError(f"Silver table(s) missing from the refresh result: {missing}")

    failed = [t for t in tables if t["status"] == "Failure"]
    for t in failed:
        error = t.get("error") or {}
        log.error(f"{t['tableName']}: {error.get('errorCode')} - {error.get('message')}")
    if failed:
        raise RuntimeError(
            f"{len(failed)} table(s) failed to sync: {[t['tableName'] for t in failed]}"
        )

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create main to run the entire process
# ────────────────────────────────────────────────────────────────────────────

def main() -> dict:
    """
    Resolve the SQL endpoint, refresh its metadata, and check every table.
    Returns:
        dict: tables_synced, tables_not_run, duration_seconds.
    """
    client = fabric.FabricRestClient()
    workspace_id = fabric.get_workspace_id()
    sql_endpoint_id = resolve_sql_endpoint_id(client, workspace_id)

    started = time.monotonic()
    tables = refresh_metadata(client, workspace_id, sql_endpoint_id)
    duration = round(time.monotonic() - started, 1)

    # lastSuccessfulSyncDateTime shows whether a NotRun table is up to date
    for t in tables:
        log.info(
            f"{t['tableName']}: {t['status']}, "
            f"last successful sync {t.get('lastSuccessfulSyncDateTime')}"
        )
    check_tables(tables)

    synced = sum(t["status"] == "Success" for t in tables)
    not_run = sum(t["status"] == "NotRun" for t in tables)
    log.info(f"Tables synced: {synced}, not run: {not_run}, duration: {duration} s")
    return {"tables_synced": synced, "tables_not_run": not_run, "duration_seconds": duration}

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Run
# ────────────────────────────────────────────────────────────────────────────
# main() runs in its own cell: exit() in the same cell drops its log output
# (lessons-learned.md, Python notebooks).

result = main()

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Exit contract — outside any try/except (ADR-009, 2026-09-30 addendum)
# ────────────────────────────────────────────────────────────────────────────
# Interactive runs: Fabric marks this cell with ✗ because exit() stops the
# notebook on purpose. Pipeline runs: the activity reports Succeeded and
# passes exitValue to the logging Script activity (ADR-009).

notebookutils.notebook.exit(json.dumps(result))